# Project 2 Hospital

In [2]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

def generate_arrival_rate (Lam):
    patients_to_time_t = np.random.poisson(lam= Lam)
    return patients_to_time_t

def generate_ward_A_length_of_stay(t):
    lam1= -(1/3650)*t**2 + (1/10)*t 
    arriaval_rate =generate_arrival_rate (lam1)
    Ward_A_lenght_of_stay = np.random.lognormal(mean=np.log(4*np.sqrt(2)),sigma=np.sqrt(np.log(2)), size=int(arriaval_rate) )
    N_patients_A = len(Ward_A_lenght_of_stay)
    return Ward_A_lenght_of_stay, N_patients_A

def generate_ward_B_length_of_stay(t):
    lam2 = 1/5*( -(1/3650)*t**2 + (1/10)*t )
    arriaval_rate = generate_arrival_rate(lam2)
    Ward_B_lenght_of_stay = np.random.lognormal(mean=np.log(6*np.sqrt(2)),sigma=np.sqrt(np.log(2)), size=int(arriaval_rate ))
    N_patients_B = len(Ward_B_lenght_of_stay)
    return Ward_B_lenght_of_stay, N_patients_B


def generate_ward_C_length_of_stay(t):
    arriaval_rate = generate_arrival_rate(6)
    Ward_C_lenght_of_stay = np.random.lognormal(mean=np.log(5*np.sqrt(2)),sigma=np.sqrt(np.log(2)), size=int(arriaval_rate ))
    N_patients_C = len(Ward_C_lenght_of_stay)
    return Ward_C_lenght_of_stay, N_patients_C



def generate_length_of_stay(t, Ward):
    if Ward=='A':
        return generate_ward_A_length_of_stay(t)
    if Ward=='B':
        return generate_ward_B_length_of_stay(t)
    if Ward=='C':
        return generate_ward_C_length_of_stay(t)


In [3]:
lenght_of_stay_C, arrivals_C = generate_length_of_stay(150,'C')
lenght_of_stay_C, arrivals_C

(array([ 8.92029143, 16.40064499,  4.35968032,  4.56664572,  4.39429149,
         3.27612747]),
 6)

In [ ]:
np.random.seed(42)
def run_Hospital_simulation(bed_distribution):
    A_beds, B_beds, C_beds = bed_distribution
    times = np.arange(1,365,1)

    # Counts everyone who tried to get into Ward X but couldnt
    blocked_WardA = 0
    blocked_WardB = 0 #tæller ikke
    blocked_WardC = 0

    # Counts everyone who was transfored to another hospital
    next_free_bed_A = np.zeros((A_beds))
    next_free_bed_B = np.zeros((B_beds))
    next_free_bed_C = np.zeros((C_beds))
    total_arrivals_list = []

    full_A = full_B = full_C = 0
    # Counts everyone who was transfored to another hospital
    relocated_A = relocated_B = relocated_C = 0
    attempts_WardA = 0

    arrivals_total_A = 0
    arrivals_total_B = 0
    arrivals_total_C = 0

    util_A = []
    util_B = []
    util_C = []

    for t in times:
        lenght_of_stay_A, arrivals_A = generate_length_of_stay(t,'A')
        lenght_of_stay_B, arrivals_B = generate_length_of_stay(t,'B')
        lenght_of_stay_C, arrivals_C = generate_length_of_stay(t,'C')
        total_arrivals = arrivals_A+arrivals_B+arrivals_C
        #print(f"t={t} Lengt of stay A:{len(lenght_of_stay_A)}, B:{len(lenght_of_stay_B)} , C:{len(lenght_of_stay_C)}")

        # estimate free beds
        available_bedsA = next_free_bed_A <= t
        available_bedsB = next_free_bed_B <= t
        available_bedsC = next_free_bed_C <= t
        arrivals_total_A += arrivals_A
        arrivals_total_B += arrivals_B
        arrivals_total_C += arrivals_C

        # kan man sorte længderne så vi blokerer beds ift længde?
        
        #_--------------------------
        # Ward B first! 
        j = -1
        indexes = np.where(available_bedsB)[0]
        admitted_B = min(len(lenght_of_stay_B), len(indexes))

        if np.any(available_bedsB) and len(lenght_of_stay_B)!=0 :
            for j,ib in enumerate(indexes):
                if j < len(lenght_of_stay_B):
                    next_free_bed_B[ib] = t+ lenght_of_stay_B[j]
                else: 
                    j -= 1
        diff_B = arrivals_B - admitted_B
        blocked_WardB += diff_B
        full_B += diff_B

        B_overflow_los = np.array([])

        if diff_B > 0:
            B_overflow_los = lenght_of_stay_B[j+1:]
            lenght_of_stay_A = np.append(lenght_of_stay_A,B_overflow_los)

        #_--------------------------
        # Ward A
        indexes = np.where(available_bedsA)[0]
        admitted_A = min(len(lenght_of_stay_A), len(indexes))
        attempts_WardA += len(lenght_of_stay_A) 

        j = -1
        if np.any(available_bedsA) and len(lenght_of_stay_A)!=0:
            for j,ia in enumerate(indexes):
                if j < len(lenght_of_stay_A):
                    next_free_bed_A[ia] = t+ lenght_of_stay_A[j]
                else: 
                    j -= 1
        diff_A = len(lenght_of_stay_A) - admitted_A
        blocked_WardA += diff_A
    
        n_B_try_A = diff_B
        n_A_try_A = arrivals_A

        A_admitted_A = min(n_A_try_A, admitted_A)
        B_admitted_A = admitted_A - A_admitted_A

        relocated_B += n_B_try_A - B_admitted_A
        relocated_A += n_A_try_A - A_admitted_A

        full_A += n_A_try_A - A_admitted_A

        #_--------------------------
        # Ward C
        indexes = np.where(available_bedsC)[0]
        admitted_C = min(len(lenght_of_stay_C), len(indexes))
        j = -1
        if np.any(available_bedsC) and len(lenght_of_stay_C)!=0:
            for j,ic in enumerate(indexes):
                if j < len(lenght_of_stay_C):
                    next_free_bed_C[ic] = t + lenght_of_stay_C[j]
                else: 
                    j -= 1
        diff_C = arrivals_C - admitted_C
        blocked_WardC += diff_C
        full_C += diff_C      
        relocated_C += diff_C

        # Update lists
        total_arrivals_list.append(total_arrivals)

        # Utilization of beds
        util_A.append(np.mean(next_free_bed_A > t))
        util_B.append(np.mean(next_free_bed_B > t))
        util_C.append(np.mean(next_free_bed_C > t))

    results = { 
        "prob_full_A": full_A / arrivals_total_A,
        "prob_full_B": full_B / arrivals_total_B,
        "prob_full_C": full_C / arrivals_total_C,

        "relocated_A": relocated_A,
        "relocated_B": relocated_B,
        "relocated_C": relocated_C,
        "relocated_total": relocated_A + relocated_B + relocated_C,

        "utilization_A": np.mean(util_A),
        "utilization_B": np.mean(util_B),
        "utilization_C": np.mean(util_C),

        "blocked_WardA": blocked_WardA,
        "blocked_WardB": blocked_WardB,
        "blocked_WardC": blocked_WardC,

        "arrivals_A": arrivals_total_A,
        "arrivals_B": arrivals_total_B,
        "arrivals_C": arrivals_total_C,

        "total_arrivals_list": total_arrivals_list
    }

    return results

    

A_beds = 30
B_beds = 15
C_beds = 30


results  = run_Hospital_simulation((A_beds,B_beds, C_beds ))


In [7]:
results['blocked_WardA'], results['utilization_A']

(1157, np.float64(0.8786630036630038))

In [9]:
def grid_search_beds(total_beds=75, n_reps=100):
    results = []

    for A_beds in range(1, total_beds):
        for B_beds in range(1, total_beds - A_beds):
            C_beds = total_beds - A_beds - B_beds

            prob_full_A_values = []
            prob_full_B_values = []
            prob_full_C_values = []

            relocated_A_values = []
            relocated_B_values = []
            relocated_C_values = []
            relocated_total_values = []

            util_A_values = []
            util_B_values = []
            util_C_values = []

            for rep in range(n_reps):
                res = run_Hospital_simulation(
                    (A_beds, B_beds, C_beds)
                )

                prob_full_A_values.append(res["prob_full_A"])
                prob_full_B_values.append(res["prob_full_B"])
                prob_full_C_values.append(res["prob_full_C"])

                relocated_A_values.append(res["relocated_A"])
                relocated_B_values.append(res["relocated_B"])
                relocated_C_values.append(res["relocated_C"])
                relocated_total_values.append(res["relocated_total"])

                util_A_values.append(res["utilization_A"])
                util_B_values.append(res["utilization_B"])
                util_C_values.append(res["utilization_C"])

            results.append({
                "A_beds": A_beds,
                "B_beds": B_beds,
                "C_beds": C_beds,

                "mean_prob_full_A": np.mean(prob_full_A_values),
                "mean_prob_full_B": np.mean(prob_full_B_values),
                "mean_prob_full_C": np.mean(prob_full_C_values),

                "mean_relocated_A": np.mean(relocated_A_values),
                "mean_relocated_B": np.mean(relocated_B_values),
                "mean_relocated_C": np.mean(relocated_C_values),
                "mean_relocated_total": np.mean(relocated_total_values),

                "sd_relocated_total": np.std(relocated_total_values, ddof=1),

                "mean_utilization_A": np.mean(util_A_values),
                "mean_utilization_B": np.mean(util_B_values),
                "mean_utilization_C": np.mean(util_C_values)
            })

    return results

In [12]:
import pandas as pd
coarse_results = grid_search_beds(total_beds=75,n_reps=1)

coarse_df = pd.DataFrame(coarse_results)

top20 = coarse_df.sort_values("mean_relocated_total").head(20)

top20

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,A_beds,B_beds,C_beds,mean_prob_full_A,mean_prob_full_B,mean_prob_full_C,mean_relocated_A,mean_relocated_B,mean_relocated_C,mean_relocated_total,sd_relocated_total,mean_utilization_A,mean_utilization_B,mean_utilization_C
2269,45,4,26,0.274085,0.761446,0.563636,569.0,218.0,1178.0,1965.0,NaN,0.845849,0.921016,0.987320
2106,40,1,34,0.368349,0.954545,0.429713,803.0,310.0,914.0,2027.0,NaN,0.869918,0.923077,0.979315
1578,27,6,42,0.539860,0.676056,0.298095,1158.0,261.0,626.0,2045.0,NaN,0.897232,0.887821,0.958006
1923,35,3,37,0.409729,0.844340,0.410593,876.0,285.0,907.0,2068.0,NaN,0.891994,0.893773,0.982031
1573,27,1,47,0.536872,0.950226,0.258741,1143.0,374.0,555.0,2072.0,NaN,0.904457,0.914835,0.962065
2173,42,1,32,0.346353,0.930131,0.486144,736.0,309.0,1035.0,2080.0,NaN,0.861983,0.950549,0.981971
2110,40,5,30,0.368397,0.703125,0.476728,809.0,258.0,1014.0,2081.0,NaN,0.844093,0.891209,0.987363
2071,39,1,35,0.366864,0.933333,0.428963,806.0,336.0,939.0,2081.0,NaN,0.892223,0.920330,0.982025
1885,34,5,36,0.442593,0.681499,0.394379,983.0,257.0,842.0,2082.0,NaN,0.880171,0.840110,0.976954
1327,22,5,48,0.595315,0.713287,0.234382,1296.0,287.0,499.0,2082.0,NaN,0.920455,0.891758,0.948088


In [ ]:
def rerun_candidates(candidates_df, n_reps=75):
    results = []

    for _, row in candidates_df.iterrows():
        A_beds = int(row["A_beds"])
        B_beds = int(row["B_beds"])
        C_beds = int(row["C_beds"])

        candidate_results = []

        for rep in range(n_reps):
            res = run_Hospital_simulation(
                (A_beds, B_beds, C_beds)
            )
            candidate_results.append(res)

        results.append({
            "A_beds": A_beds,
            "B_beds": B_beds,
            "C_beds": C_beds,

            "mean_prob_full_A": np.mean([r["prob_full_A"] for r in candidate_results]),
            "mean_prob_full_B": np.mean([r["prob_full_B"] for r in candidate_results]),
            "mean_prob_full_C": np.mean([r["prob_full_C"] for r in candidate_results]),

            "mean_relocated_A": np.mean([r["relocated_A"] for r in candidate_results]),
            "mean_relocated_B": np.mean([r["relocated_B"] for r in candidate_results]),
            "mean_relocated_C": np.mean([r["relocated_C"] for r in candidate_results]),
            "mean_relocated_total": np.mean([r["relocated_total"] for r in candidate_results]),

            "sd_relocated_total": np.std(
                [r["relocated_total"] for r in candidate_results],
                ddof=1
            ),

            "mean_utilization_A": np.mean([r["utilization_A"] for r in candidate_results]),
            "mean_utilization_B": np.mean([r["utilization_B"] for r in candidate_results]),
            "mean_utilization_C": np.mean([r["utilization_C"] for r in candidate_results])
        })

    return pd.DataFrame(results)



,A_beds,B_beds,C_beds,mean_prob_full_A,mean_prob_full_B,mean_prob_full_C,mean_relocated_A,mean_relocated_B,mean_relocated_C,mean_relocated_total,sd_relocated_total,mean_utilization_A,mean_utilization_B,mean_utilization_C
0,45,4,26,0.318194,0.759425,0.577035,707.40,254.27,1263.75,2225.42,77.151848,0.845711,0.891065,0.989330
1,40,1,34,0.381458,0.938467,0.455267,847.08,325.67,998.27,2171.02,65.799509,0.872944,0.925220,0.981972
2,27,6,42,0.543698,0.653602,0.336684,1207.40,268.14,736.41,2211.95,61.935022,0.898664,0.867738,0.971014
3,35,3,37,0.437045,0.820900,0.409021,969.65,309.33,892.80,2171.78,71.308064,0.880485,0.904240,0.977718
4,27,1,47,0.547457,0.937945,0.261910,1212.91,372.76,572.92,2158.59,67.968218,0.911620,0.923819,0.960433
5,42,1,32,0.359562,0.938565,0.488128,799.52,317.09,1067.67,2184.28,75.625884,0.867357,0.928901,0.983925
6,40,5,30,0.368125,0.706987,0.518389,816.56,254.29,1130.26,2201.11,74.279692,0.857434,0.874522,0.985692
7,39,1,35,0.396928,0.935871,0.441405,882.22,329.32,965.74,2177.28,85.162125,0.877498,0.930659,0.980557
8,34,5,36,0.448798,0.709547,0.426477,1000.85,275.32,935.14,2211.31,77.770537,0.879558,0.880747,0.979506
9,22,5,48,0.618063,0.702491,0.251420,1372.72,293.76,551.05,2217.53,74.749339,0.916774,0.878780,0.958794


In [14]:
# find bedste candi
final_df = rerun_candidates(top20, n_reps=75)
final_best = final_df.loc[final_df["mean_relocated_total"].idxmin()]
final_best

A_beds                    32.000000
B_beds                     2.000000
C_beds                    41.000000
mean_prob_full_A           0.477296
mean_prob_full_B           0.875707
mean_prob_full_C           0.349500
mean_relocated_A        1056.613333
mean_relocated_B         333.093333
mean_relocated_C         762.546667
mean_relocated_total    2152.253333
sd_relocated_total        83.993902
mean_utilization_A         0.893907
mean_utilization_B         0.915403
mean_utilization_C         0.973029
Name: 18, dtype: float64

In [15]:
final_df

,A_beds,B_beds,C_beds,mean_prob_full_A,mean_prob_full_B,mean_prob_full_C,mean_relocated_A,mean_relocated_B,mean_relocated_C,mean_relocated_total,sd_relocated_total,mean_utilization_A,mean_utilization_B,mean_utilization_C
0,45,4,26,0.317245,0.758943,0.582131,702.240000,251.613333,1274.746667,2228.600000,76.174941,0.843491,0.895311,0.988862
1,40,1,34,0.380089,0.937426,0.452240,845.653333,324.853333,988.213333,2158.720000,92.807205,0.871680,0.926630,0.981267
2,27,6,42,0.545720,0.653629,0.336879,1212.120000,270.946667,736.786667,2219.853333,69.391699,0.899464,0.872473,0.971371
3,35,3,37,0.440790,0.819829,0.410475,979.133333,307.773333,898.280000,2185.186667,77.949688,0.879960,0.903187,0.978579
4,27,1,47,0.548856,0.937473,0.263408,1217.173333,375.320000,577.080000,2169.573333,79.526670,0.911533,0.925568,0.961632
5,42,1,32,0.358967,0.938163,0.489417,798.240000,319.106667,1072.253333,2189.600000,78.752778,0.864706,0.924725,0.984136
6,40,5,30,0.374033,0.705600,0.517778,830.560000,254.053333,1130.786667,2215.400000,78.457770,0.860360,0.878821,0.985822
7,39,1,35,0.394534,0.937100,0.440643,875.280000,324.533333,961.440000,2161.253333,85.354443,0.874719,0.927289,0.980842
8,34,5,36,0.448535,0.708702,0.425721,992.613333,272.413333,930.480000,2195.506667,74.216151,0.875168,0.878557,0.979359
9,22,5,48,0.618596,0.706432,0.249871,1373.400000,300.600000,546.200000,2220.200000,88.501947,0.915528,0.884278,0.958460


In [23]:
# kører det hele igen for at få alle paramentre ud
n_reps = 100

all_results = []

for _ in range(n_reps):
    all_results.append(
        run_Hospital_simulation((32,2,41))
    )
summary = {
    key: np.mean([r[key] for r in all_results])
    for key in [
        "prob_full_A",
        "prob_full_B",
        "prob_full_C",
        "relocated_A",
        "relocated_B",
        "relocated_C",
        "relocated_total",
        "utilization_A",
        "utilization_B",
        "utilization_C"
    ]
}

summary

{'prob_full_A': np.float64(0.47629176720337774),
 'prob_full_B': np.float64(0.8772589966158222),
 'prob_full_C': np.float64(0.3514699232730187),
 'relocated_A': np.float64(1053.09),
 'relocated_B': np.float64(332.87),
 'relocated_C': np.float64(767.95),
 'relocated_total': np.float64(2153.91),
 'utilization_A': np.float64(0.8934177541208791),
 'utilization_B': np.float64(0.9131593406593406),
 'utilization_C': np.float64(0.9718440096488876)}

In [24]:
# Probability of full ward on arrival
summary['prob_full_A' ], summary['prob_full_B' ], summary['prob_full_C' ]

(np.float64(0.47629176720337774),
 np.float64(0.8772589966158222),
 np.float64(0.3514699232730187))

In [25]:
# mean relocated patiens
summary['relocated_A' ], summary['relocated_B' ], summary['relocated_C' ], summary['relocated_total']

(np.float64(1053.09),
 np.float64(332.87),
 np.float64(767.95),
 np.float64(2153.91))

In [26]:
# mean utilization
summary['utilization_A' ], summary['utilization_B' ], summary['utilization_C' ]

(np.float64(0.8934177541208791),
 np.float64(0.9131593406593406),
 np.float64(0.9718440096488876))